# 01 · EDA y limpieza

Datos: fraud_oracle.csv ([Kaggle](https://www.kaggle.com/datasets/shivamb/vehicle-claim-fraud-detection)), 15.420 reclamaciones. Objetivo: limpiar los datos y ver qué variables aportan señal.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/fraud_oracle.csv')
print(df.shape)
print('Nulos:', df.isna().sum().sum(), '| Duplicados:', df.duplicated().sum())
df.head()

(15420, 33)
Nulos: 0 | Duplicados: 0


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


## 1. Variable objetivo

In [2]:
df['FraudFound_P'].value_counts().to_frame().assign(pct=lambda t: (t['count'] / t['count'].sum() * 100).round(1))

,count,pct
FraudFound_P,,
0,14497,94.0
1,923,6.0


Solo el 6 % es fraude. Un modelo que prediga siempre no fraude acertaría el 94 % sin detectar nada, así que la accuracy no sirve. Métricas del proyecto: PR-AUC (principal, el azar da 0,06), recall en el top-k % de reclamaciones y ROC-AUC (secundaria, se infla con clases desbalanceadas).

## 2. Limpieza

### 2.1 Age = 0

In [3]:
print('Filas con Age = 0:', (df['Age'] == 0).sum())
print(df.loc[df['Age'] == 0, 'AgeOfPolicyHolder'].value_counts())
df.groupby('AgeOfPolicyHolder')['Age'].agg(['min', 'max'])

Filas con Age = 0: 320
AgeOfPolicyHolder
16 to 17    320
Name: count, dtype: int64


,min,max
AgeOfPolicyHolder,,
16 to 17,0,0
18 to 20,16,17
21 to 25,18,20
26 to 30,21,25
31 to 35,26,35
36 to 40,36,45
41 to 50,46,55
51 to 65,56,65
over 65,66,80


Las 320 filas con Age = 0 son exactamente las de AgeOfPolicyHolder = 16 to 17: el 0 es un código de desconocido, no una edad.

Decisión: convertir Age = 0 en NaN. Inventar un valor sesgaría el modelo y eliminar Age perdería detalle en el resto de filas. La regresión logística imputará la mediana dentro del pipeline y los árboles manejan los NaN.

In [4]:
df['Age'] = df['Age'].replace(0, np.nan)
df['Age'].isna().sum()

np.int64(320)

### 2.2 Reclamación sin fecha

In [5]:
print(df['DayOfWeekClaimed'].value_counts())
print(df['MonthClaimed'].value_counts())

DayOfWeekClaimed
Monday       3757
Tuesday      3375
Wednesday    2951
Thursday     2660
Friday       2497
Saturday      127
Sunday         52
0               1
Name: count, dtype: int64
MonthClaimed
Jan    1446
May    1411
Mar    1348
Oct    1339
Jun    1293
Feb    1287
Nov    1285
Apr    1271
Sep    1242
Jul    1225
Dec    1146
Aug    1126
0         1
Name: count, dtype: int64


Fila con valor 0:

In [6]:
df[df['MonthClaimed'] == '0'].T

,1516
Month,Jul
WeekOfMonth,2
DayOfWeek,Monday
Make,Honda
AccidentArea,Rural
DayOfWeekClaimed,0
MonthClaimed,0
WeekOfMonthClaimed,1
Sex,Male
MaritalStatus,Single


Es una sola fila, sin día ni mes de reclamación y con Days_Policy_Claim = none.

Decisión: eliminarla, un único caso no permite aprender nada.

In [7]:
df = df[df['MonthClaimed'] != '0']
df.shape

(15419, 33)

### 2.3 Marcas (Make)

In [8]:
df['Make'].value_counts()

Make
Pontiac      3837
Toyota       3121
Honda        2800
Mazda        2354
Chevrolet    1681
Accura        472
Ford          450
VW            283
Dodge         109
Saab          108
Mercury        83
Saturn         58
Nisson         30
BMW            15
Jaguar          6
Porche          5
Mecedes         4
Ferrari         2
Lexus           1
Name: count, dtype: int64

Erratas: Accura, Mecedes, Nisson, Porche. ¿Conviven con la grafía correcta?

In [9]:
df['Make'].isin(['Acura', 'Mercedes', 'Nissan', 'Porsche']).sum()

np.int64(0)

Ninguna coincide: cada marca tiene una sola grafía, así que las erratas no parten categorías.

Decisión: corregirlas para que el análisis sea legible.

In [10]:
df['Make'] = df['Make'].replace({'Accura': 'Acura', 'Mecedes': 'Mercedes', 'Nisson': 'Nissan', 'Porche': 'Porsche'})
df.groupby('Make')['FraudFound_P'].agg(n='size', tasa_fraude='mean').sort_values('n')

,n,tasa_fraude
Make,,
Lexus,1,0.000000
Ferrari,2,0.000000
Mercedes,4,0.250000
Porsche,5,0.000000
Jaguar,6,0.000000
BMW,15,0.066667
Nissan,30,0.033333
Saturn,58,0.103448
Mercury,83,0.072289


Marcas con muy pocas reclamaciones: intervalo de confianza al 95 % de su tasa de fraude.

In [11]:
from scipy.stats import binomtest

marcas = df.groupby('Make')['FraudFound_P'].agg(n='size', fraudes='sum').sort_values('n')
ic = [binomtest(f, n).proportion_ci() for f, n in zip(marcas['fraudes'], marcas['n'])]
marcas['tasa'] = (marcas['fraudes'] / marcas['n']).round(3)
marcas['ic_min'] = [round(i.low, 3) for i in ic]
marcas['ic_max'] = [round(i.high, 3) for i in ic]
marcas

,n,fraudes,tasa,ic_min,ic_max
Make,,,,,
Lexus,1,0,0.000,0.000,0.975
Ferrari,2,0,0.000,0.000,0.842
Mercedes,4,1,0.250,0.006,0.806
Porsche,5,0,0.000,0.000,0.522
Jaguar,6,0,0.000,0.000,0.459
BMW,15,1,0.067,0.002,0.319
Nissan,30,1,0.033,0.001,0.172
Saturn,58,6,0.103,0.039,0.212
Mercury,83,6,0.072,0.027,0.151


Con n = 30 (Nissan) la tasa real puede estar entre el 0,1 % y el 17 %: no informa nada. A partir de unas 100 reclamaciones los intervalos ya son útiles.

Decisión: agrupar en Otras las categorías con menos de 100 reclamaciones (Lexus, Ferrari, Mercedes, Porsche, Jaguar, BMW, Nissan, Saturn y Mercury). Se hará dentro del pipeline con solo los datos de train para evitar fugas.

## 3. Qué variables aportan señal

### 3.1 Ranking con la V de Cramér

0 = sin relación con el fraude, 1 = lo predice por completo.

In [12]:
from scipy.stats.contingency import association

columnas = df.columns.drop(['FraudFound_P', 'PolicyNumber', 'Age'])
v_cramer = pd.Series({c: association(pd.crosstab(df[c], df['FraudFound_P'])) for c in columnas})
v_cramer.sort_values(ascending=False).round(3)

PolicyType              0.168
BasePolicy              0.162
VehicleCategory         0.137
Fault                   0.131
AddressChange_Claim     0.082
Deductible              0.069
VehiclePrice            0.066
Make                    0.062
PastNumberOfClaims      0.059
MonthClaimed            0.052
AgeOfPolicyHolder       0.046
Month                   0.044
AgeOfVehicle            0.038
NumberOfSuppliments     0.034
AccidentArea            0.034
Sex                     0.030
RepNumber               0.028
Days_Policy_Accident    0.027
DayOfWeek               0.026
Year                    0.025
AgentType               0.023
DayOfWeekClaimed        0.018
Days_Policy_Claim       0.018
PoliceReportFiled       0.016
WeekOfMonthClaimed      0.015
WeekOfMonth             0.013
NumberOfCars            0.013
DriverRating            0.010
MaritalStatus           0.008
WitnessPresent          0.008
dtype: float64

Todas las asociaciones son débiles o moderadas (V ≤ 0,17): ninguna variable predice el fraude por sí sola. Destacan PolicyType, BasePolicy, VehicleCategory y Fault.

### 3.2 Posible fuga de datos

Variables que podrían conocerse después de abrir la reclamación: NumberOfSuppliments (se acumulan después), PoliceReportFiled (la denuncia puede llegar tarde) y Fault (la determina el perito).

In [13]:
for c in ['Fault', 'PoliceReportFiled', 'NumberOfSuppliments']:
    print(df.groupby(c)['FraudFound_P'].agg(n='size', tasa='mean').round(3))
    print()

                   n   tasa
Fault                      
Policy Holder  11229  0.079
Third Party     4190  0.009

                       n   tasa
PoliceReportFiled              
No                 14991  0.061
Yes                  428  0.037

                        n   tasa
NumberOfSuppliments             
1 to 2               2489  0.064
3 to 5               2017  0.048
more than 5          3867  0.050
none                 7046  0.067



NumberOfSuppliments y PoliceReportFiled apenas tienen señal (V = 0,034 y 0,016). Fault sí (7,9 % de fraude frente a 0,9 %).

Decisión: eliminar las dos primeras y mantener Fault como supuesto, comparando el modelo con y sin ella.

In [14]:
df = df.drop(columns=['NumberOfSuppliments', 'PoliceReportFiled'])
df.shape

(15419, 31)

### 3.3 Redundancias

PolicyType frente a VehicleCategory + BasePolicy.

In [15]:
(df['PolicyType'] == df['VehicleCategory'] + ' - ' + df['BasePolicy']).mean()

np.float64(0.6765678708087425)

In [16]:
pd.crosstab(df['PolicyType'], df['VehicleCategory'] + ' - ' + df['BasePolicy'])

col_0,Sedan - All Perils,Sedan - Collision,Sport - All Perils,Sport - Collision,Sport - Liability,Utility - All Perils,Utility - Collision,Utility - Liability
PolicyType,,,,,,,,
Sedan - All Perils,4086,0,0,0,0,0,0,0
Sedan - Collision,0,5584,0,0,0,0,0,0
Sedan - Liability,0,0,0,0,4987,0,0,0
Sport - All Perils,0,0,22,0,0,0,0,0
Sport - Collision,0,0,0,348,0,0,0,0
Sport - Liability,0,0,0,0,1,0,0,0
Utility - All Perils,0,0,0,0,0,340,0,0
Utility - Collision,0,0,0,0,0,0,30,0
Utility - Liability,0,0,0,0,0,0,0,21


La correspondencia es uno a uno salvo en 1 fila: contienen la misma información. ¿Cuántas categorías quedarían por debajo de 100 reclamaciones con cada opción?

In [17]:
for c in ['PolicyType', 'VehicleCategory', 'BasePolicy']:
    print(df.groupby(c)['FraudFound_P'].agg(n='size', tasa='mean').round(3))
    print()

                         n   tasa
PolicyType                       
Sedan - All Perils    4086  0.101
Sedan - Collision     5584  0.069
Sedan - Liability     4987  0.007
Sport - All Perils      22  0.000
Sport - Collision      348  0.138
Sport - Liability        1  0.000
Utility - All Perils   340  0.121
Utility - Collision     30  0.100
Utility - Liability     21  0.000

                    n   tasa
VehicleCategory             
Sedan            9670  0.082
Sport            5358  0.016
Utility           391  0.113

               n   tasa
BasePolicy             
All Perils  4448  0.102
Collision   5962  0.073
Liability   5009  0.007



Con PolicyType, cuatro categorías tienen menos de 100 casos y al agruparlas se mezclarían tasas muy distintas. Con VehicleCategory y BasePolicy todas superan los 100.

Decisión: eliminar PolicyType.

Limitación: en 4.987 pólizas Liability, PolicyType dice Sedan y VehicleCategory dice Sport; una de las dos está mal etiquetada.

In [18]:
df = df.drop(columns=['PolicyType'])
df.shape

(15419, 30)

### 3.4 Variables temporales

Tasa de fraude por año.

In [19]:
df.groupby('Year')['FraudFound_P'].agg(n='size', tasa='mean').round(3)

,n,tasa
Year,,
1994,6141,0.067
1995,5195,0.058
1996,4083,0.052


La tasa apenas varía (V = 0,025) y el modelo se aplicará a años futuros que no ha visto.

Decisión: eliminar Year.

In [20]:
df = df.drop(columns=['Year'])
df.shape

(15419, 29)

Retraso entre accidente y reclamación (aproximado: solo hay mes).

In [21]:
meses = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
retraso = (df['MonthClaimed'].map(meses.index) - df['Month'].map(meses.index)) % 12
df.groupby(retraso)['FraudFound_P'].agg(n='size', tasa='mean').round(3)

,n,tasa
0,11442,0.055
1,3211,0.070
2,390,0.105
3,145,0.076
4,80,0.075
5,38,0.079
6,37,0.054
7,29,0.000
8,23,0.130
9,10,0.000


La tasa sube con el retraso (5,5 % → 7,0 % → 10,5 %) y con 3 meses o más hay pocos casos.

Decisión: crear retraso_meses agrupando 3 o más en un mismo valor.

In [22]:
df['retraso_meses'] = ((df['MonthClaimed'].map(meses.index) - df['Month'].map(meses.index)) % 12).clip(upper=3)
df.groupby('retraso_meses')['FraudFound_P'].agg(n='size', tasa='mean').round(3)

,n,tasa
retraso_meses,,
0,11442,0.055
1,3211,0.070
2,390,0.105
3,376,0.066


### 3.5 Identificador

PolicyNumber es único por fila y no aporta información: se elimina.

In [23]:
print(df['PolicyNumber'].is_unique)
df = df.drop(columns=['PolicyNumber'])
df.shape

True


(15419, 29)

## 4. Guardado

Dataset limpio en data/processed/. La selección de variables y la agrupación de categorías raras se hacen en el notebook 02 solo con train.

In [24]:
df.to_csv('../data/processed/reclamaciones_limpio.csv', index=False)
df.shape

(15419, 29)